# Dataset unificado (CAP + Sleep-EDF + Scoring)

Primer paso de la tesis: combina las **tres bases polisomnográficas** —CAP, Sleep-EDF y Scoring— en un solo conjunto con esquema homogéneo. Es la cohorte de sujetos sanos sobre la que se apoya todo el análisis posterior.

Lee los tres CSV de épocas con sus archivos demográficos y escribe la carpeta **dataset/**. Es idempotente: puede correrse las veces que se quiera sin efectos colaterales.

**Salidas:**
- **epocas_unificado.csv** — una fila por época de 30 s (dataset, paciente, epoca, tiempo, fase_texto, fase_num).
- **pacientes_unificado.csv** — una fila por paciente (dataset, paciente, sexo, edad, n_epocas, duracion_h).
- **README.md** — diccionario de columnas y mapeo de fases.

**Fases (igual en las tres bases):** 0 = Vigilia, 1 = S1, 2 = S2, 3 = SWS (fusión de S3 y S4, criterio de Rechtschaffen y Kales), 4 = REM, 5 = Sin clasificar.

In [1]:
import re
from pathlib import Path
import pandas as pd

RAIZ = Path("..").resolve()
DATOS = RAIZ / "Datos"
SRC_CAP = DATOS / "datos_internet/cap/epocas_cap.csv"
SRC_EDF = DATOS / "datos_internet/sleep_edf/epocas_sleep_edf.csv"
SRC_SCO = DATOS / "datos_internet/scoring/epocas_scoring.csv"
META_CAP = DATOS / "datos_internet/cap/gender-age.xlsx"
META_EDF = DATOS / "datos_internet/sleep_edf/SC-subjects.xls"

SALIDA = RAIZ / "dataset"
SALIDA.mkdir(parents=True, exist_ok=True)
print("Salida:", SALIDA)

Salida: /home/penguin/Documentos/el-lenguaje-del-sueno/dataset


## 1. Combinar las épocas

Las tres bases ya comparten el mismo esquema y sus IDs vienen prefijados (**CAP_**, **EDF_**, **SCO_**), así que basta con apilarlas y añadir la columna **dataset**, que marca el origen de cada fila.

In [2]:
frames = []
for tag, src in [("CAP", SRC_CAP), ("EDF", SRC_EDF), ("SCO", SRC_SCO)]:
    df = pd.read_csv(src)
    df.insert(0, "dataset", tag)
    frames.append(df)

epocas = pd.concat(frames, ignore_index=True)
print("Total épocas:", len(epocas))
print("Pacientes por dataset:", epocas.groupby("dataset")["paciente"].nunique().to_dict())
epocas.head()

Total épocas: 112809
Pacientes por dataset: {'CAP': 16, 'EDF': 82, 'SCO': 29}


,dataset,paciente,epoca,tiempo,fase_texto,fase_num
0,CAP,CAP_S01_N1,1,22:09:33,Vigilia,0
1,CAP,CAP_S01_N1,2,22:10:03,Vigilia,0
2,CAP,CAP_S01_N1,3,22:10:33,Vigilia,0
3,CAP,CAP_S01_N1,4,22:11:03,Vigilia,0
4,CAP,CAP_S01_N1,5,22:11:33,Vigilia,0


## 2. Construir la tabla de pacientes con demografía

Cada base guarda la demografía en un formato distinto; aquí se normaliza todo a una sola tabla.

- **CAP** — archivo **gender-age.xlsx** (sin encabezado: id, sexo, edad). Se conservan los 16 sujetos sanos (los **N1…N16**), renombrados a **CAP_S01_N1 … CAP_S16_N1**.
- **Sleep-EDF** — archivo **SC-subjects.xls** (sujeto, noche, edad, sexo). Se renombra a **EDF_Sxx_Ny** según el orden de aparición del sujeto, conservando solo registros de 65 años o menos.
- **Scoring** — no tiene demografía pública, así que **sexo** y **edad** quedan vacíos.

In [3]:
# CAP: nos quedamos con los 16 sujetos sanos (los de tipo 'N')
cap_raw = pd.read_excel(META_CAP, header=None, names=["id_raw", "sexo", "edad"])
cap_raw["tipo"] = cap_raw["id_raw"].astype(str).str.extract(r"^([A-Za-z]+)")
cap_raw["num"]  = cap_raw["id_raw"].astype(str).str.extract(r"(\d+)$")
cap_sanos = cap_raw[cap_raw["tipo"] == "N"].copy()
cap_sanos["num"] = cap_sanos["num"].astype(int)
cap_sanos = cap_sanos.sort_values("num").reset_index(drop=True)
cap_sanos["paciente"] = [f"CAP_S{i+1:02d}_N1" for i in range(len(cap_sanos))]
cap_meta = cap_sanos[["paciente", "sexo", "edad"]].copy()
cap_meta["dataset"] = "CAP"
print("CAP demografía:", len(cap_meta))
cap_meta.head()

CAP demografía: 16


,paciente,sexo,edad,dataset
0,CAP_S01_N1,F,37,CAP
1,CAP_S02_N1,M,34,CAP
2,CAP_S03_N1,F,35,CAP
3,CAP_S04_N1,F,25,CAP
4,CAP_S05_N1,F,35,CAP


In [4]:
# EDF: filtra a 65 años o menos y renombra cada sujeto a EDF_Sxx_Ny por orden de aparición
edf_raw = pd.read_excel(META_EDF)
edf_raw = edf_raw.rename(columns={"sex (F=1)": "sexo_cod"})
edf_raw["sexo"] = edf_raw["sexo_cod"].map({1: "F", 2: "M"})
edf_raw = edf_raw[edf_raw["age"] <= 65].copy()
# subject_id -> Sxx, numerados según el orden del sujeto
subjects_unicos = sorted(edf_raw["subject"].unique())
subject_to_idx = {s: i + 1 for i, s in enumerate(subjects_unicos)}
edf_raw["paciente"] = edf_raw.apply(
    lambda r: f"EDF_S{subject_to_idx[r['subject']]:02d}_N{int(r['night'])}", axis=1
)
edf_meta = edf_raw[["paciente", "sexo", "age"]].rename(columns={"age": "edad"})
edf_meta["dataset"] = "EDF"
print("EDF demografía:", len(edf_meta))
edf_meta.head()

EDF demografía: 82


,paciente,sexo,edad,dataset
0,EDF_S01_N1,F,33,EDF
1,EDF_S01_N2,F,33,EDF
2,EDF_S02_N1,F,33,EDF
3,EDF_S02_N2,F,33,EDF
4,EDF_S03_N1,F,26,EDF


In [5]:
# SCO: no hay demografía pública, sexo y edad quedan vacíos
sco_ids = sorted(epocas.loc[epocas["dataset"] == "SCO", "paciente"].unique())
sco_meta = pd.DataFrame({
    "paciente": sco_ids,
    "sexo": pd.NA,
    "edad": pd.NA,
    "dataset": "SCO",
})
print("SCO demografía:", len(sco_meta))
sco_meta.head()

SCO demografía: 29


,paciente,sexo,edad,dataset
0,SCO_AA_N1,<NA>,<NA>,SCO
1,SCO_AG_N1,<NA>,<NA>,SCO
2,SCO_AQ_N1,<NA>,<NA>,SCO
3,SCO_AR_N1,<NA>,<NA>,SCO
4,SCO_AR_N2,<NA>,<NA>,SCO


In [6]:
demo = pd.concat([cap_meta, edf_meta, sco_meta], ignore_index=True)

# Conteos por paciente a partir de las épocas (120 épocas de 30 s = 1 hora)
stats = (
    epocas.groupby("paciente")
    .agg(n_epocas=("epoca", "count"))
    .reset_index()
)
stats["duracion_h"] = (stats["n_epocas"] / 120.0).round(2)

pacientes = (
    demo.merge(stats, on="paciente", how="right")
    .loc[:, ["dataset", "paciente", "sexo", "edad", "n_epocas", "duracion_h"]]
    .sort_values(["dataset", "paciente"])
    .reset_index(drop=True)
)
print("Total pacientes:", len(pacientes))
print("Faltan sexo:", pacientes["sexo"].isna().sum(), " | Faltan edad:", pacientes["edad"].isna().sum())
pacientes.head()

Total pacientes: 127
Faltan sexo: 29  | Faltan edad: 29


,dataset,paciente,sexo,edad,n_epocas,duracion_h
0,CAP,CAP_S01_N1,F,37,1140,9.50
1,CAP,CAP_S02_N1,M,34,999,8.32
2,CAP,CAP_S03_N1,F,35,999,8.32
3,CAP,CAP_S04_N1,F,25,979,8.16
4,CAP,CAP_S05_N1,F,35,1007,8.39


## 3. Guardar los CSVs unificados

In [7]:
out_epocas = SALIDA / "epocas_unificado.csv"
out_pacientes = SALIDA / "pacientes_unificado.csv"
epocas.to_csv(out_epocas, index=False)
pacientes.to_csv(out_pacientes, index=False)
print("Escrito:", out_epocas, f"({out_epocas.stat().st_size/1e6:.2f} MB)")
print("Escrito:", out_pacientes, f"({out_pacientes.stat().st_size/1e3:.2f} KB)")

Escrito: /home/penguin/Documentos/el-lenguaje-del-sueno/dataset/epocas_unificado.csv (3.79 MB)
Escrito: /home/penguin/Documentos/el-lenguaje-del-sueno/dataset/pacientes_unificado.csv (3.62 KB)


## 4. Auditoría final

In [8]:
print("=== epocas_unificado.csv ===")
print("  filas:", len(epocas), " cols:", list(epocas.columns))
print("  NaNs:", epocas.isna().sum().sum())
print("  por dataset:")
print(epocas.groupby("dataset").agg(pacientes=("paciente", "nunique"), epocas=("epoca", "count")))
print("\n  distribución de fases (% por dataset):")
pct = (
    epocas.groupby(["dataset", "fase_texto"]).size().unstack(fill_value=0)
    .pipe(lambda d: d.div(d.sum(axis=1), axis=0) * 100).round(2)
)
print(pct)

print("\n=== pacientes_unificado.csv ===")
print("  filas:", len(pacientes))
print(pacientes.groupby("dataset").agg(
    n=("paciente", "count"),
    edad_media=("edad", "mean"),
    horas_total=("duracion_h", "sum"),
))

=== epocas_unificado.csv ===
  filas: 112809  cols: ['dataset', 'paciente', 'epoca', 'tiempo', 'fase_texto', 'fase_num']
  NaNs: 0
  por dataset:


         pacientes  epocas
dataset                   
CAP             16   15947
EDF             82   70410
SCO             29   26452

  distribución de fases (% por dataset):


fase_texto    REM    S1     S2    SWS  Sin_clasificar  Vigilia
dataset                                                       
CAP         22.74  2.90  40.67  24.26            0.04     9.39
EDF         20.53  7.91  51.67  12.06            0.12     7.70
SCO         17.01  7.05  40.87  19.48            6.67     8.93



=== pacientes_unificado.csv ===
  filas: 127
          n edad_media  horas_total
dataset                            
CAP      16    32.1875       132.88
EDF      82  41.878049       586.78
SCO      29        NaN       220.44


## 5. README de la carpeta unificada

In [9]:
readme = SALIDA / "README.md"
readme.write_text("""# Dataset unificado

Carpeta generada por `notebooks/1_construir_dataset.ipynb`.
Combina los tres datasets (CAP, Sleep-EDF, Scoring) en archivos únicos con esquema homogéneo.

## Archivos

### `epocas_unificado.csv`
Una fila por época (30 s).

| columna     | tipo    | descripción                                                     |
|-------------|---------|-----------------------------------------------------------------|
| dataset     | str     | `CAP`, `EDF` o `SCO`                                            |
| paciente    | str     | ID estilo CAP (`CAP_S01_N1`, `EDF_S01_N1`, `SCO_AA_N1`, …)      |
| epoca       | int     | 1..N consecutiva dentro del paciente                            |
| tiempo      | str     | `HH:MM:SS` reloj real desde el inicio del registro              |
| fase_texto  | str     | `Vigilia`, `S1`, `S2`, `SWS`, `REM`, `Sin_clasificar`           |
| fase_num    | int     | 0..5 (corresponde 1-a-1 con `fase_texto`)                       |

### `pacientes_unificado.csv`
Una fila por paciente.

| columna    | tipo  | descripción                                                |
|------------|-------|------------------------------------------------------------|
| dataset    | str   | `CAP`, `EDF` o `SCO`                                       |
| paciente   | str   | ID del paciente                                            |
| sexo       | str   | `M`/`F` (NaN en Scoring)                                   |
| edad       | int   | años (NaN en Scoring)                                      |
| n_epocas   | int   | número total de épocas                                     |
| duracion_h | float | n_epocas / 120                                             |

## Mapeo de fases (común a los 3 datasets)

| fase_num | fase_texto       | nota                              |
|----------|------------------|-----------------------------------|
| 0        | Vigilia          |                                   |
| 1        | S1               |                                   |
| 2        | S2               |                                   |
| 3        | SWS              | fusión de S3 y S4 (clásico R&K)   |
| 4        | REM              |                                   |
| 5        | Sin_clasificar   | incluye Movimiento y `?`          |

## Reproducir

```
jupyter nbconvert --to notebook --execute --inplace \\
  notebooks/1_construir_dataset.ipynb
```

## Fuentes

- `Datos/datos_internet/cap/epocas_cap.csv` + `gender-age.xlsx`
- `Datos/datos_internet/sleep_edf/epocas_sleep_edf.csv` + `SC-subjects.xls`
- `Datos/datos_internet/scoring/epocas_scoring.csv` (sin demografía)
""")
print("Escrito:", readme)

Escrito: /home/penguin/Documentos/el-lenguaje-del-sueno/dataset/README.md
